## Order & Product Profitability Risk Modeling

### Notebook 3 — Feature Engineering & Target Definition

#### Objective
Prepare the dataset for modeling by defining the target variable (profit risk) and creating meaningful features.

#### Key Tasks
- Define profit risk (target variable)
- Create new features based on business logic
- Ensure no data leakage
- Prepare final dataset for modeling

In [18]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/orders_clean.csv")
df.head()

,Order Id,total_sales,total_order_value,total_discount,total_quantity,order_profit,Customer Segment,Market,Shipping Mode,Order Region,Order Country
0,1,299.980011,239.979996,60.000000,1,88.790001,Consumer,LATAM,Standard Class,Central America,México
1,2,579.980011,529.380005,50.600000,7,195.900002,Consumer,LATAM,Standard Class,South America,Colombia
2,4,699.850010,620.870014,78.980000,14,124.090000,Home Office,LATAM,Standard Class,South America,Colombia
3,5,1129.860039,987.070007,142.789999,10,390.089995,Consumer,LATAM,Standard Class,South America,Colombia
4,7,579.920013,525.520004,54.400000,7,203.929998,Consumer,LATAM,Second Class,South America,Brasil


### 1. Defining Profit Risk (Target Variable)

The goal is to predict whether an order is risky (low or negative profit).

Instead of using only loss-making orders, I define risk as the bottom 25% of orders based on profit. This helps capture both loss-making and very low-profit orders.

In [3]:
profit_threshold = df['order_profit'].quantile(0.25)
print("Profit threshold (25th percentile):", profit_threshold)

Profit threshold (25th percentile): 7.79750037325


In [4]:
df['profit_risk'] = (df['order_profit'] <= profit_threshold).astype(int)

In [5]:
df['profit_risk'].value_counts(normalize=True)

profit_risk
0    0.75
1    0.25
Name: proportion, dtype: float64

In [6]:
df.groupby('profit_risk')['order_profit'].describe()

,count,mean,std,min,25%,50%,75%,max
profit_risk,,,,,,,,
0,49314.0,133.279122,100.408279,7.800000,55.089994,110.710001,188.330001,980.280006
1,16438.0,-158.512207,223.314486,-4325.849979,-218.940006,-76.824996,-14.430000,7.790001


**Observation:**
The threshold successfully separates low and negative profit orders from high-profit ones. Risky orders have significantly lower average and median profit, confirming that the target variable captures meaningful business risk.

### 2. Preparing Features for Modeling


In [7]:
features = [
    'total_sales',
    'total_discount',
    'total_quantity',
    'Customer Segment',
    'Market',
    'Order Region',
    'Shipping Mode'
]

In [8]:
model_df = df[features + ['profit_risk']].copy()

In [9]:
model_df.head()

,total_sales,total_discount,total_quantity,Customer Segment,Market,Order Region,Shipping Mode,profit_risk
0,299.980011,60.000000,1,Consumer,LATAM,Central America,Standard Class,0
1,579.980011,50.600000,7,Consumer,LATAM,South America,Standard Class,0
2,699.850010,78.980000,14,Home Office,LATAM,South America,Standard Class,0
3,1129.860039,142.789999,10,Consumer,LATAM,South America,Standard Class,0
4,579.920013,54.400000,7,Consumer,LATAM,South America,Second Class,0


In [10]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65752 entries, 0 to 65751
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   total_sales       65752 non-null  float64
 1   total_discount    65752 non-null  float64
 2   total_quantity    65752 non-null  int64  
 3   Customer Segment  65752 non-null  object 
 4   Market            65752 non-null  object 
 5   Order Region      65752 non-null  object 
 6   Shipping Mode     65752 non-null  object 
 7   profit_risk       65752 non-null  int32  
dtypes: float64(2), int32(1), int64(1), object(4)
memory usage: 3.8+ MB


In [11]:
model_df_encoded = pd.get_dummies(
    model_df,
    columns=['Customer Segment', 'Market', 'Order Region', 'Shipping Mode'],
    drop_first=True
)

In [12]:
bool_cols = model_df_encoded.select_dtypes(include='bool').columns

model_df_encoded[bool_cols] = model_df_encoded[bool_cols].astype(int)

In [13]:
model_df_encoded.dtypes

total_sales                     float64
total_discount                  float64
total_quantity                    int64
profit_risk                       int32
Customer Segment_Corporate        int32
Customer Segment_Home Office      int32
Market_Europe                     int32
Market_LATAM                      int32
Market_Pacific Asia               int32
Market_USCA                       int32
Order Region_Caribbean            int32
Order Region_Central Africa       int32
Order Region_Central America      int32
Order Region_Central Asia         int32
Order Region_East Africa          int32
Order Region_East of USA          int32
Order Region_Eastern Asia         int32
Order Region_Eastern Europe       int32
Order Region_North Africa         int32
Order Region_Northern Europe      int32
Order Region_Oceania              int32
Order Region_South America        int32
Order Region_South Asia           int32
Order Region_South of  USA        int32
Order Region_Southeast Asia       int32


### 3. Feature Engineering


In [14]:
model_df_encoded['discount_pct'] = (
    model_df_encoded['total_discount'] / model_df_encoded['total_sales']
)

In [16]:
model_df_encoded['avg_price_per_item'] = (
    model_df_encoded['total_sales'] / model_df_encoded['total_quantity']
)

In [21]:
model_df_encoded['discount_per_item'] = (
    model_df_encoded['total_discount'] / model_df_encoded['total_quantity']
)

In [22]:
model_df_encoded = model_df_encoded.replace([np.inf, -np.inf], 0)
model_df_encoded = model_df_encoded.fillna(0)

In [23]:
model_df_encoded.describe()

,total_sales,total_discount,total_quantity,profit_risk,Customer Segment_Corporate,Customer Segment_Home Office,Market_Europe,Market_LATAM,Market_Pacific Asia,Market_USCA,...,Order Region_West Africa,Order Region_West Asia,Order Region_West of USA,Order Region_Western Europe,Shipping Mode_Same Day,Shipping Mode_Second Class,Shipping Mode_Standard Class,discount_pct,avg_price_per_item,discount_per_item
count,65752.000000,65752.000000,65752.000000,65752.000000,65752.000000,65752.000000,65752.000000,65752.000000,65752.000000,65752.000000,...,65752.000000,65752.000000,65752.000000,65752.000000,65752.00000,65752.000000,65752.000000,65752.000000,65752.000000,65752.000000
mean,559.446633,56.734067,5.841328,0.250000,0.301983,0.179112,0.282288,0.261300,0.267323,0.130475,...,0.018600,0.030752,0.040562,0.152239,0.05431,0.194336,0.598065,0.101528,134.698220,13.671861
std,355.537119,46.584295,4.169632,0.433016,0.459122,0.383449,0.450116,0.439346,0.442566,0.336828,...,0.135109,0.172646,0.197274,0.359255,0.22663,0.395692,0.490293,0.054173,149.488131,20.151591
min,9.990000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,9.990000,0.000000
25%,265.957504,20.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.058994,59.990002,4.970000
50%,504.890008,47.390000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,1.000000,0.099983,91.988877,8.583333
75%,799.950028,82.390002,9.000000,0.250000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,1.000000,0.138583,149.980002,14.990416
max,3449.909988,681.500000,24.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,0.250433,1500.000000,375.000000


**Observation:**
The newly created features show reasonable ranges with no invalid or extreme values. Discount percentage remains within expected limits, and price-related features capture variation in order structure.

In [26]:
model_df_encoded.to_csv("../data/processed/model_data.csv", index=False)

## Notebook Summary

### What was done

- Defined target variable (profit risk using bottom 25% profit)
- Selected relevant business features
- Removed leakage variables
- Applied one-hot encoding for categorical variables
- Created new features:
  - Discount percentage
  - Average price per item
  - Discount per item

---

### Key Learnings

- Feature engineering improves representation of business behavior
- Relative measures (like percentage) are often more meaningful than absolute values
- Clean and well-structured data is critical before modeling

---

### Next Step

The dataset is now ready for modeling. In the next notebook, we will build and evaluate machine learning models to predict profitability risk.